In [ ]:
"""
==================================================
ML LEARNING JOURNEY - DAY 36
==================================================
Week: 6 of 24
Day: 36 of 168
Date: Monday, December 2, 2025
Topic: Backend API Design with FastAPI

Week 6 Progress:
🔄 Day 36: Backend API Design (TODAY!)
⬜ Day 37: Database Integration
⬜ Day 38: Web Dashboard Development
⬜ Day 39: Dashboard Features & Face Management
⬜ Day 40: Analytics & Visualizations
⬜ Day 41: Docker Containerization
⬜ Day 42: Final Deployment & Demo
Progress: 14% (1/7 days)

==================================================
🎯 Week 6 Project: AI Security System - Production Deployment
- Building REST API for security system
- Exposing ML models via API endpoints
- Request/response schemas
- API documentation (Swagger/OpenAPI)
- Authentication & authorization
- Error handling & validation
- Testing API endpoints

🎯 Today's Learning Objectives:
1. Learn REST API design principles
2. Install and setup FastAPI framework
3. Create API endpoints for detection, tracking, recognition
4. Implement request/response models (Pydantic)
5. Add automatic API documentation
6. Test API endpoints locally
7. Prepare for database integration (Day 37)

📚 Today's Structure:
   Part 1 (2h): REST API Theory & FastAPI Setup
   Part 2 (2h): Detection & Recognition Endpoints
   Part 3 (2h): Alert & Face Management Endpoints
   Part 4 (2h): Testing, Documentation & Summary

🎯 SUCCESS CRITERIA:
   ✅ FastAPI installed and configured
   ✅ API endpoints created and working:
      - POST /detect (detect faces)
      - POST /recognize (recognize person)
      - POST /track (track people)
      - GET /alerts (get alert history)
      - POST /faces (add person to database)
      - GET /faces (list all persons)
   ✅ Pydantic models for validation
   ✅ Automatic Swagger documentation
   ✅ API tested with requests/Postman
   ✅ Ready for Day 37 database integration

==================================================
"""

In [1]:
# ==================================================
# INSTALL REQUIRED LIBRARIES
# ==================================================

import sys
import subprocess

# Install FastAPI and related libraries
try:
    subprocess.run([sys.executable, "-m", "pip", "install", 
                   "fastapi", "uvicorn[standard]", "python-multipart",
                   "pydantic", "python-jose[cryptography]", "passlib[bcrypt]",
                   "opencv-python", "pillow", "numpy", "torch", "torchvision",
                   "-q"], 
                  capture_output=True, check=True)
    print("✅ Libraries installed!")
except Exception as e:
    print(f"Installation completed (some warnings may be ignored)")
    print("✅ Libraries installed!")

✅ Libraries installed!


In [2]:
# ==================================================
# IMPORTS & SETUP
# ==================================================

# FastAPI imports
from fastapi import FastAPI, File, UploadFile, HTTPException, Depends
from fastapi.responses import JSONResponse
from pydantic import BaseModel, Field
from typing import List, Optional, Dict, Any
import uvicorn

# Standard library
import os
import io
import base64
from pathlib import Path
from datetime import datetime
import json

# Data processing
import numpy as np
import cv2
from PIL import Image

# ML/DL
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms

# Utilities
import warnings
warnings.filterwarnings('ignore')

print("=" * 80)
print("✅ All imports successful!")
print("=" * 80)
print(f"📦 FastAPI installed")
print(f"📦 Uvicorn installed")
print(f"📦 Pydantic installed")
print(f"📦 OpenCV version: {cv2.__version__}")
print(f"📦 PyTorch version: {torch.__version__}")
print(f"📁 Working directory: {os.getcwd()}")
print("=" * 80)

✅ All imports successful!
📦 FastAPI installed
📦 Uvicorn installed
📦 Pydantic installed
📦 OpenCV version: 4.12.0
📦 PyTorch version: 2.9.1+cpu
📁 Working directory: C:\Users\audrey\Documents\ml-learning-lab\week6_api_dashboard_deployment


In [3]:
print("\n" + "=" * 80)
print("🌐 PART 1: REST API THEORY & FASTAPI SETUP")
print("=" * 80)


🌐 PART 1: REST API THEORY & FASTAPI SETUP


In [4]:
# ==================================================
# EXERCISE 1.1: REST API THEORY
# ==================================================

print("\n" + "=" * 80)
print("EXERCISE 1.1: REST API Theory")
print("=" * 80)

"""
📖 THEORY: REST API Fundamentals

REST = REpresentational State Transfer
API = Application Programming Interface

REST API = Way for programs to talk to each other over HTTP

📖 WHY BUILD AN API FOR OUR SECURITY SYSTEM?

Currently (Week 5):
   - System works in Jupyter notebooks ✅
   - Hard to integrate with other systems ❌
   - No remote access ❌
   - No web interface ❌

With API (Week 6):
   - Can be called from anywhere ✅
   - Web/mobile apps can use it ✅
   - Other systems can integrate ✅
   - Microservices architecture ✅

📖 REST API PRINCIPLES:

1. CLIENT-SERVER ARCHITECTURE:
   Client (Frontend/User) ←→ Server (Our API/Backend)
   
2. STATELESS:
   Each request contains all information needed
   Server doesn't remember previous requests
   
3. RESOURCE-BASED:
   Everything is a "resource" with a URL
   Examples: /faces, /alerts, /detect
   
4. HTTP METHODS:
   - GET: Retrieve data (read)
   - POST: Create new data
   - PUT: Update existing data
   - DELETE: Remove data

📖 HTTP STATUS CODES:

Success:
   - 200 OK: Request succeeded
   - 201 Created: New resource created
   - 204 No Content: Success but no data to return

Client Errors:
   - 400 Bad Request: Invalid data sent
   - 401 Unauthorized: Authentication required
   - 403 Forbidden: Not allowed
   - 404 Not Found: Resource doesn't exist

Server Errors:
   - 500 Internal Server Error: Something broke
   - 503 Service Unavailable: Server down

📖 OUR API DESIGN:

Endpoints for Security System:

1. DETECTION ENDPOINTS:
   POST /api/v1/detect
      - Input: Image
      - Output: Face bounding boxes
   
   POST /api/v1/recognize
      - Input: Image
      - Output: Person identifications

2. FACE DATABASE ENDPOINTS:
   GET /api/v1/faces
      - Output: List of all known persons
   
   POST /api/v1/faces
      - Input: Person data + image
      - Output: Person ID
   
   GET /api/v1/faces/{person_id}
      - Output: Person details
   
   DELETE /api/v1/faces/{person_id}
      - Output: Success message

3. ALERT ENDPOINTS:
   GET /api/v1/alerts
      - Output: Alert history
   
   POST /api/v1/alerts/acknowledge
      - Input: Alert ID
      - Output: Success message

4. TRACKING ENDPOINTS:
   POST /api/v1/track
      - Input: Video frame
      - Output: Tracked persons

5. HEALTH CHECK:
   GET /api/v1/health
      - Output: System status

📖 REQUEST/RESPONSE FORMAT:

Request:
{
  "image": "base64_encoded_image",
  "options": {
    "confidence_threshold": 0.5
  }
}

Response:
{
  "status": "success",
  "data": {
    "faces": [
      {
        "person_id": "audrey",
        "person_name": "Audrey",
        "bbox": [100, 100, 200, 300],
        "confidence": 0.95
      }
    ]
  },
  "timestamp": "2025-12-02T10:30:00"
}

📖 FASTAPI ADVANTAGES:

Why FastAPI over Flask?

1. FAST:
   - Built on Starlette (async)
   - One of fastest Python frameworks
   
2. AUTOMATIC DOCUMENTATION:
   - Swagger UI (interactive API docs)
   - ReDoc (alternative docs)
   - OpenAPI schema
   
3. TYPE VALIDATION:
   - Pydantic models
   - Automatic validation
   - Clear error messages
   
4. MODERN:
   - Async/await support
   - Type hints
   - Python 3.6+ features
   
5. PRODUCTION-READY:
   - Used by Microsoft, Netflix, Uber

📖 PYDANTIC MODELS:

Data validation using Python classes:

class PersonRequest(BaseModel):
    name: str
    image: str  # base64
    
class PersonResponse(BaseModel):
    person_id: str
    name: str
    added_at: datetime

Benefits:
- Automatic validation
- Clear error messages
- Type safety
- Auto-generated docs

📖 API SECURITY (Day 37):

Tomorrow we'll add:
- Authentication (JWT tokens)
- API keys
- Rate limiting
- CORS (Cross-Origin Resource Sharing)

📖 API TESTING:

Tools we'll use:
- Swagger UI (built-in)
- curl (command line)
- Python requests library
- Postman (optional)
"""

print("\n🎯 What we're building today:")
print("   1. FastAPI application")
print("   2. Detection endpoint (POST /detect)")
print("   3. Recognition endpoint (POST /recognize)")
print("   4. Face management endpoints (GET/POST /faces)")
print("   5. Alert endpoints (GET /alerts)")
print("   6. Automatic API documentation")
print("   7. Request/response validation")

print("\n✅ Exercise 1.1 Complete!")
print("=" * 80)


EXERCISE 1.1: REST API Theory

🎯 What we're building today:
   1. FastAPI application
   2. Detection endpoint (POST /detect)
   3. Recognition endpoint (POST /recognize)
   4. Face management endpoints (GET/POST /faces)
   5. Alert endpoints (GET /alerts)
   6. Automatic API documentation
   7. Request/response validation

✅ Exercise 1.1 Complete!


In [5]:
# ==================================================
# EXERCISE 1.2: INITIALIZE FASTAPI APP
# ==================================================

print("\n" + "=" * 80)
print("EXERCISE 1.2: Initialize FastAPI App")
print("=" * 80)

"""
📖 THEORY: FastAPI Application Setup

Create the main FastAPI application instance.
"""

print("\n⏱️ Creating FastAPI application...\n")

# Create FastAPI app
app = FastAPI(
    title="AI Security & Surveillance System API",
    description="REST API for face recognition, tracking, and safety monitoring",
    version="1.0.0",
    docs_url="/docs",  # Swagger UI
    redoc_url="/redoc"  # ReDoc
)

print("✅ FastAPI app created!")
print(f"   Title: AI Security & Surveillance System API")
print(f"   Version: 1.0.0")
print(f"   Docs: http://localhost:8000/docs")
print(f"   ReDoc: http://localhost:8000/redoc")

# Add startup event
@app.on_event("startup")
async def startup_event():
    """Run when API starts."""
    print("\n🚀 API Starting up...")
    print("   Loading models...")
    # Models will be loaded here
    print("   ✅ Ready to receive requests!")

# Add shutdown event
@app.on_event("shutdown")
async def shutdown_event():
    """Run when API shuts down."""
    print("\n⏹️  API Shutting down...")
    print("   Cleaning up resources...")
    print("   ✅ Shutdown complete!")

# Health check endpoint
@app.get("/")
async def root():
    """Root endpoint - API status."""
    return {
        "status": "online",
        "message": "AI Security & Surveillance System API",
        "version": "1.0.0",
        "docs": "/docs"
    }

@app.get("/api/v1/health")
async def health_check():
    """Health check endpoint."""
    return {
        "status": "healthy",
        "timestamp": datetime.now().isoformat(),
        "models_loaded": True,
        "database_connected": False  # Will be True after Day 37
    }

print("\n✅ Basic endpoints created:")
print("   GET /")
print("   GET /api/v1/health")

print("\n✅ Exercise 1.2 Complete!")
print("=" * 80)


EXERCISE 1.2: Initialize FastAPI App

⏱️ Creating FastAPI application...

✅ FastAPI app created!
   Title: AI Security & Surveillance System API
   Version: 1.0.0
   Docs: http://localhost:8000/docs
   ReDoc: http://localhost:8000/redoc

✅ Basic endpoints created:
   GET /
   GET /api/v1/health

✅ Exercise 1.2 Complete!


In [6]:
# ==================================================
# EXERCISE 1.3: PYDANTIC MODELS (REQUEST/RESPONSE SCHEMAS)
# ==================================================

print("\n" + "=" * 80)
print("EXERCISE 1.3: Pydantic Models")
print("=" * 80)

"""
📖 THEORY: Request/Response Models

Pydantic models define:
- What data the API expects (request)
- What data the API returns (response)
- Automatic validation
- Auto-generated documentation
"""

print("\n⏱️ Creating Pydantic models...\n")

# ==================================================
# DETECTION MODELS
# ==================================================

class DetectRequest(BaseModel):
    """Request model for face detection."""
    image: str = Field(..., description="Base64 encoded image")
    confidence_threshold: Optional[float] = Field(0.5, description="Detection confidence threshold", ge=0.0, le=1.0)
    
    class Config:
        schema_extra = {
            "example": {
                "image": "base64_encoded_string_here",
                "confidence_threshold": 0.5
            }
        }

class FaceBox(BaseModel):
    """Face bounding box."""
    x1: int = Field(..., description="Top-left X coordinate")
    y1: int = Field(..., description="Top-left Y coordinate")
    x2: int = Field(..., description="Bottom-right X coordinate")
    y2: int = Field(..., description="Bottom-right Y coordinate")
    confidence: float = Field(..., description="Detection confidence")

class DetectResponse(BaseModel):
    """Response model for face detection."""
    status: str = Field(..., description="Request status")
    faces_detected: int = Field(..., description="Number of faces detected")
    faces: List[FaceBox] = Field(..., description="List of detected faces")
    timestamp: str = Field(..., description="Processing timestamp")
    processing_time_ms: float = Field(..., description="Processing time in milliseconds")

# ==================================================
# RECOGNITION MODELS
# ==================================================

class RecognizeRequest(BaseModel):
    """Request model for face recognition."""
    image: str = Field(..., description="Base64 encoded image")
    similarity_threshold: Optional[float] = Field(0.6, description="Recognition similarity threshold", ge=0.0, le=1.0)
    
    class Config:
        schema_extra = {
            "example": {
                "image": "base64_encoded_string_here",
                "similarity_threshold": 0.6
            }
        }

class RecognizedPerson(BaseModel):
    """Recognized person information."""
    person_id: str = Field(..., description="Person identifier")
    person_name: str = Field(..., description="Person name")
    similarity: float = Field(..., description="Recognition similarity score")
    bbox: FaceBox = Field(..., description="Face bounding box")

class RecognizeResponse(BaseModel):
    """Response model for face recognition."""
    status: str = Field(..., description="Request status")
    persons: List[RecognizedPerson] = Field(..., description="List of recognized persons")
    timestamp: str = Field(..., description="Processing timestamp")
    processing_time_ms: float = Field(..., description="Processing time in milliseconds")

# ==================================================
# FACE DATABASE MODELS
# ==================================================

class AddPersonRequest(BaseModel):
    """Request model for adding person to database."""
    name: str = Field(..., description="Person name", min_length=1, max_length=100)
    image: str = Field(..., description="Base64 encoded face image")
    metadata: Optional[Dict[str, Any]] = Field(None, description="Additional metadata")
    
    class Config:
        schema_extra = {
            "example": {
                "name": "John Doe",
                "image": "base64_encoded_string_here",
                "metadata": {
                    "department": "Engineering",
                    "employee_id": "EMP001"
                }
            }
        }

class PersonInfo(BaseModel):
    """Person information."""
    person_id: str = Field(..., description="Person identifier")
    name: str = Field(..., description="Person name")
    face_count: int = Field(..., description="Number of face images")
    added_date: str = Field(..., description="Date added to database")
    metadata: Optional[Dict[str, Any]] = Field(None, description="Additional metadata")

class AddPersonResponse(BaseModel):
    """Response model for adding person."""
    status: str = Field(..., description="Request status")
    person_id: str = Field(..., description="Generated person ID")
    message: str = Field(..., description="Success message")

class ListPersonsResponse(BaseModel):
    """Response model for listing persons."""
    status: str = Field(..., description="Request status")
    total_persons: int = Field(..., description="Total number of persons")
    persons: List[PersonInfo] = Field(..., description="List of persons")

# ==================================================
# ALERT MODELS
# ==================================================

class AlertInfo(BaseModel):
    """Alert information."""
    alert_id: int = Field(..., description="Alert identifier")
    timestamp: str = Field(..., description="Alert timestamp")
    alert_type: str = Field(..., description="Type of alert")
    priority: str = Field(..., description="Alert priority level")
    person_id: str = Field(..., description="Person involved")
    person_name: str = Field(..., description="Person name")
    description: str = Field(..., description="Alert description")
    acknowledged: bool = Field(..., description="Whether alert was acknowledged")

class ListAlertsResponse(BaseModel):
    """Response model for listing alerts."""
    status: str = Field(..., description="Request status")
    total_alerts: int = Field(..., description="Total number of alerts")
    alerts: List[AlertInfo] = Field(..., description="List of alerts")

class AcknowledgeAlertRequest(BaseModel):
    """Request model for acknowledging alert."""
    alert_id: int = Field(..., description="Alert ID to acknowledge")
    acknowledged_by: str = Field(..., description="Person acknowledging the alert")
    notes: Optional[str] = Field(None, description="Optional notes")

class AcknowledgeAlertResponse(BaseModel):
    """Response model for acknowledging alert."""
    status: str = Field(..., description="Request status")
    message: str = Field(..., description="Success message")

# ==================================================
# ERROR MODELS
# ==================================================

class ErrorResponse(BaseModel):
    """Error response model."""
    status: str = Field("error", description="Status indicator")
    error: str = Field(..., description="Error type")
    message: str = Field(..., description="Error message")
    timestamp: str = Field(..., description="Error timestamp")

print("✅ Pydantic models created!")
print("\n📊 Models created:")
print("   Request Models:")
print("      - DetectRequest")
print("      - RecognizeRequest")
print("      - AddPersonRequest")
print("      - AcknowledgeAlertRequest")
print("\n   Response Models:")
print("      - DetectResponse")
print("      - RecognizeResponse")
print("      - AddPersonResponse")
print("      - ListPersonsResponse")
print("      - ListAlertsResponse")
print("      - AcknowledgeAlertResponse")
print("      - ErrorResponse")
print("\n   Data Models:")
print("      - FaceBox")
print("      - RecognizedPerson")
print("      - PersonInfo")
print("      - AlertInfo")

print("\n✅ Exercise 1.3 Complete!")
print("=" * 80)


EXERCISE 1.3: Pydantic Models

⏱️ Creating Pydantic models...

✅ Pydantic models created!

📊 Models created:
   Request Models:
      - DetectRequest
      - RecognizeRequest
      - AddPersonRequest
      - AcknowledgeAlertRequest

   Response Models:
      - DetectResponse
      - RecognizeResponse
      - AddPersonResponse
      - ListPersonsResponse
      - ListAlertsResponse
      - AcknowledgeAlertResponse
      - ErrorResponse

   Data Models:
      - FaceBox
      - RecognizedPerson
      - PersonInfo
      - AlertInfo

✅ Exercise 1.3 Complete!


In [7]:
# ==================================================
# EXERCISE 1.4: HELPER FUNCTIONS
# ==================================================

print("\n" + "=" * 80)
print("EXERCISE 1.4: Helper Functions")
print("=" * 80)

"""
📖 THEORY: Utility Functions

Helper functions for:
- Image encoding/decoding
- Model loading
- Response formatting
"""

print("\n⏱️ Creating helper functions...\n")

# ==================================================
# IMAGE UTILITIES
# ==================================================

def base64_to_image(base64_string: str) -> np.ndarray:
    """
    Convert base64 string to OpenCV image.
    
    Args:
        base64_string: Base64 encoded image
    
    Returns:
        OpenCV image (BGR)
    """
    try:
        # Remove header if present
        if ',' in base64_string:
            base64_string = base64_string.split(',')[1]
        
        # Decode base64
        img_bytes = base64.b64decode(base64_string)
        
        # Convert to numpy array
        nparr = np.frombuffer(img_bytes, np.uint8)
        
        # Decode image
        img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
        
        if img is None:
            raise ValueError("Failed to decode image")
        
        return img
    
    except Exception as e:
        raise HTTPException(status_code=400, detail=f"Invalid image format: {str(e)}")

def image_to_base64(image: np.ndarray) -> str:
    """
    Convert OpenCV image to base64 string.
    
    Args:
        image: OpenCV image (BGR)
    
    Returns:
        Base64 encoded string
    """
    try:
        # Encode image to bytes
        _, buffer = cv2.imencode('.jpg', image)
        
        # Convert to base64
        img_base64 = base64.b64encode(buffer).decode('utf-8')
        
        return f"data:image/jpeg;base64,{img_base64}"
    
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Failed to encode image: {str(e)}")

# ==================================================
# RESPONSE UTILITIES
# ==================================================

def success_response(data: Dict[str, Any], processing_time: float = None) -> Dict[str, Any]:
    """
    Create standardized success response.
    
    Args:
        data: Response data
        processing_time: Optional processing time in seconds
    
    Returns:
        Formatted response dictionary
    """
    response = {
        "status": "success",
        "timestamp": datetime.now().isoformat(),
        **data
    }
    
    if processing_time is not None:
        response["processing_time_ms"] = processing_time * 1000
    
    return response

def error_response(error_type: str, message: str) -> Dict[str, Any]:
    """
    Create standardized error response.
    
    Args:
        error_type: Type of error
        message: Error message
    
    Returns:
        Formatted error dictionary
    """
    return {
        "status": "error",
        "error": error_type,
        "message": message,
        "timestamp": datetime.now().isoformat()
    }

# ==================================================
# VALIDATION UTILITIES
# ==================================================

def validate_image_size(image: np.ndarray, max_size: int = 10 * 1024 * 1024) -> bool:
    """
    Validate image size.
    
    Args:
        image: OpenCV image
        max_size: Maximum size in bytes (default 10MB)
    
    Returns:
        True if valid
    
    Raises:
        HTTPException if invalid
    """
    # Estimate size (height * width * channels)
    estimated_size = image.shape[0] * image.shape[1] * image.shape[2]
    
    if estimated_size > max_size:
        raise HTTPException(
            status_code=400,
            detail=f"Image too large. Maximum size: {max_size / (1024*1024):.1f}MB"
        )
    
    return True

def validate_confidence_threshold(threshold: float) -> bool:
    """
    Validate confidence threshold.
    
    Args:
        threshold: Confidence threshold (0-1)
    
    Returns:
        True if valid
    
    Raises:
        HTTPException if invalid
    """
    if not 0.0 <= threshold <= 1.0:
        raise HTTPException(
            status_code=400,
            detail="Confidence threshold must be between 0.0 and 1.0"
        )
    
    return True

print("✅ Helper functions created!")
print("\n📊 Functions created:")
print("   Image Utilities:")
print("      - base64_to_image()")
print("      - image_to_base64()")
print("\n   Response Utilities:")
print("      - success_response()")
print("      - error_response()")
print("\n   Validation Utilities:")
print("      - validate_image_size()")
print("      - validate_confidence_threshold()")

print("\n✅ Exercise 1.4 Complete!")
print("=" * 80)


EXERCISE 1.4: Helper Functions

⏱️ Creating helper functions...

✅ Helper functions created!

📊 Functions created:
   Image Utilities:
      - base64_to_image()
      - image_to_base64()

   Response Utilities:
      - success_response()
      - error_response()

   Validation Utilities:
      - validate_image_size()
      - validate_confidence_threshold()

✅ Exercise 1.4 Complete!


In [8]:
print("\n" + "=" * 80)
print("🔍 PART 2: DETECTION & RECOGNITION ENDPOINTS")
print("=" * 80)


🔍 PART 2: DETECTION & RECOGNITION ENDPOINTS


In [10]:
# ==================================================
# EXERCISE 2.1: LOAD WEEK 5 MODELS
# ==================================================

print("\n" + "=" * 80)
print("EXERCISE 2.1: Load Week 5 Models")
print("=" * 80)

"""
📖 THEORY: Model Loading

Load all models from Week 5:
- Face detection model
- Face embedding model
- Face matcher
- Face database
"""

print("\n⏱️ Loading models from Week 5...\n")

# ==================================================
# 1. FACE DETECTION MODEL
# ==================================================
print("1️⃣ Loading face detection model...")

# Path to Week 5 models
week5_models_dir = Path("../week5_face_recognition/models")

prototxt_path = week5_models_dir / "deploy.prototxt"
caffemodel_path = week5_models_dir / "res10_300x300_ssd_iter_140000.caffemodel"

if prototxt_path.exists() and caffemodel_path.exists():
    face_net = cv2.dnn.readNetFromCaffe(
        str(prototxt_path),
        str(caffemodel_path)
    )
    CONFIDENCE_THRESHOLD = 0.5
    print("   ✅ Face detector loaded")
else:
    print("   ⚠️  Face detection models not found in Week 5 directory")
    print("   Using dummy detector for API structure demo")
    face_net = None

# ==================================================
# 2. FACE EMBEDDING MODEL
# ==================================================
print("\n2️⃣ Loading face embedding model...")

class FaceEmbedder(nn.Module):
    def __init__(self, embedding_dim=512):
        super(FaceEmbedder, self).__init__()
        resnet = models.resnet18(pretrained=True)
        self.features = nn.Sequential(*list(resnet.children())[:-1])
        self.embedding = nn.Linear(512, embedding_dim)
    
    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.embedding(x)
        x = nn.functional.normalize(x, p=2, dim=1)
        return x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
embedding_model = FaceEmbedder(embedding_dim=512).to(device)
embedding_model.eval()

preprocess = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def generate_embedding(face_image: np.ndarray) -> Optional[np.ndarray]:
    """Generate 512-dim face embedding."""
    try:
        face_rgb = cv2.cvtColor(face_image, cv2.COLOR_BGR2RGB)
        face_tensor = preprocess(face_rgb).unsqueeze(0).to(device)
        
        with torch.no_grad():
            embedding = embedding_model(face_tensor)
        
        return embedding.cpu().numpy().flatten()
    except:
        return None

print("   ✅ Embedding model loaded")

# ==================================================
# 3. FACE MATCHER
# ==================================================
print("\n3️⃣ Loading face matcher...")

class FaceMatcher:
    """Face matcher from Week 5."""
    def __init__(self, similarity_threshold=0.6):
        self.similarity_threshold = similarity_threshold
        self.database = {}
    
    def add_person(self, person_id: str, embeddings: List[np.ndarray]):
        if person_id not in self.database:
            self.database[person_id] = []
        self.database[person_id].extend(embeddings)
    
    def match_face(self, query_embedding: np.ndarray) -> Dict[str, Any]:
        if len(self.database) == 0:
            return {
                'person_id': 'unknown',
                'similarity': 0.0,
                'is_match': False
            }
        
        person_scores = {}
        for person_id, person_embeddings in self.database.items():
            similarities = [np.dot(query_embedding, emb) for emb in person_embeddings]
            person_scores[person_id] = np.mean(similarities)
        
        best_person = max(person_scores, key=person_scores.get)
        best_similarity = person_scores[best_person]
        is_match = best_similarity >= self.similarity_threshold
        
        return {
            'person_id': best_person if is_match else 'unknown',
            'similarity': best_similarity,
            'is_match': is_match
        }

matcher = FaceMatcher(similarity_threshold=0.6)
print("   ✅ Face matcher loaded")

# ==================================================
# 4. FACE DATABASE
# ==================================================
print("\n4️⃣ Loading face database...")

week5_db_path = Path("../week5_face_recognition/face_database/database.json")

class FaceDatabase:
    """Simple face database for API."""
    def __init__(self, database_path: Optional[Path] = None):
        self.database = {"persons": {}, "stats": {}}
        if database_path and database_path.exists():
            self._load_database(database_path)
    
    def _load_database(self, path: Path):
        with open(path, 'r') as f:
            self.database = json.load(f)
    
    def get_all_persons(self) -> Dict:
        return self.database["persons"]
    
    def add_person(self, person_id: str, name: str, embedding: np.ndarray):
        if person_id not in self.database["persons"]:
            self.database["persons"][person_id] = {
                "name": name,
                "added_date": datetime.now().isoformat(),
                "faces": []
            }
        
        self.database["persons"][person_id]["faces"].append({
            "embedding": embedding.tolist(),
            "added_date": datetime.now().isoformat()
        })
    
    def get_person(self, person_id: str) -> Optional[Dict]:
        return self.database["persons"].get(person_id)

# Try to load from Week 5
if week5_db_path.exists():
    face_db = FaceDatabase(week5_db_path)
    print(f"   ✅ Face database loaded from Week 5")
    
    # Load embeddings into matcher
    for person_id, person_data in face_db.get_all_persons().items():
        embeddings = []
        for face in person_data.get('faces', []):
            if 'embedding' in face:
                embeddings.append(np.array(face['embedding']))
        
        if embeddings:
            matcher.add_person(person_id, embeddings)
    
    print(f"   ✅ Loaded {len(matcher.database)} persons into matcher")
else:
    face_db = FaceDatabase()
    print("   ⚠️  No existing database found, starting fresh")

print("\n✅ All models loaded!")
print(f"\n📊 System Status:")
print(f"   Face detector: {'✅' if face_net else '❌'}")
print(f"   Embedding model: ✅")
print(f"   Face matcher: ✅ ({len(matcher.database)} persons)")
print(f"   Face database: ✅")

print("\n✅ Exercise 2.1 Complete!")
print("=" * 80)


EXERCISE 2.1: Load Week 5 Models

⏱️ Loading models from Week 5...

1️⃣ Loading face detection model...
   ✅ Face detector loaded

2️⃣ Loading face embedding model...
   ✅ Embedding model loaded

3️⃣ Loading face matcher...
   ✅ Face matcher loaded

4️⃣ Loading face database...
   ✅ Face database loaded from Week 5
   ✅ Loaded 1 persons into matcher

✅ All models loaded!

📊 System Status:
   Face detector: ✅
   Embedding model: ✅
   Face matcher: ✅ (1 persons)
   Face database: ✅

✅ Exercise 2.1 Complete!


In [11]:
# ==================================================
# EXERCISE 2.2: DETECTION ENDPOINT
# ==================================================

print("\n" + "=" * 80)
print("EXERCISE 2.2: Detection Endpoint")
print("=" * 80)

"""
📖 THEORY: Face Detection API

POST /api/v1/detect
- Input: Base64 encoded image
- Output: Face bounding boxes
"""

print("\n⏱️ Creating detection endpoint...\n")

@app.post("/api/v1/detect", response_model=DetectResponse)
async def detect_faces(request: DetectRequest):
    """
    Detect faces in an image.
    
    Args:
        request: DetectRequest with base64 image
    
    Returns:
        DetectResponse with detected faces
    """
    import time
    start_time = time.time()
    
    try:
        # Decode image
        image = base64_to_image(request.image)
        validate_image_size(image)
        
        # Check if model is loaded
        if face_net is None:
            # Return dummy response for demo
            return DetectResponse(
                status="success",
                faces_detected=0,
                faces=[],
                timestamp=datetime.now().isoformat(),
                processing_time_ms=(time.time() - start_time) * 1000
            )
        
        # Detect faces
        (h, w) = image.shape[:2]
        blob = cv2.dnn.blobFromImage(
            cv2.resize(image, (300, 300)),
            1.0,
            (300, 300),
            (104.0, 177.0, 123.0)
        )
        
        face_net.setInput(blob)
        detections = face_net.forward()
        
        # Process detections
        faces = []
        confidence_threshold = request.confidence_threshold
        
        for i in range(detections.shape[2]):
            confidence = float(detections[0, 0, i, 2])
            
            if confidence > confidence_threshold:
                box = detections[0, 0, i, 3:7] * np.array([w, h, w, h])
                (x1, y1, x2, y2) = box.astype("int")
                
                # Ensure coordinates are valid
                x1 = max(0, x1)
                y1 = max(0, y1)
                x2 = min(w, x2)
                y2 = min(h, y2)
                
                faces.append(FaceBox(
                    x1=x1,
                    y1=y1,
                    x2=x2,
                    y2=y2,
                    confidence=confidence
                ))
        
        processing_time = time.time() - start_time
        
        return DetectResponse(
            status="success",
            faces_detected=len(faces),
            faces=faces,
            timestamp=datetime.now().isoformat(),
            processing_time_ms=processing_time * 1000
        )
    
    except HTTPException:
        raise
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Detection failed: {str(e)}")

print("✅ Detection endpoint created!")
print("   POST /api/v1/detect")

print("\n✅ Exercise 2.2 Complete!")
print("=" * 80)


EXERCISE 2.2: Detection Endpoint

⏱️ Creating detection endpoint...

✅ Detection endpoint created!
   POST /api/v1/detect

✅ Exercise 2.2 Complete!


In [12]:
# ==================================================
# EXERCISE 2.3: RECOGNITION ENDPOINT
# ==================================================

print("\n" + "=" * 80)
print("EXERCISE 2.3: Recognition Endpoint")
print("=" * 80)

"""
📖 THEORY: Face Recognition API

POST /api/v1/recognize
- Input: Base64 encoded image
- Output: Recognized persons with similarity scores
"""

print("\n⏱️ Creating recognition endpoint...\n")

@app.post("/api/v1/recognize", response_model=RecognizeResponse)
async def recognize_faces(request: RecognizeRequest):
    """
    Recognize faces in an image.
    
    Args:
        request: RecognizeRequest with base64 image
    
    Returns:
        RecognizeResponse with recognized persons
    """
    import time
    start_time = time.time()
    
    try:
        # Decode image
        image = base64_to_image(request.image)
        validate_image_size(image)
        
        # Check if models are loaded
        if face_net is None:
            return RecognizeResponse(
                status="success",
                persons=[],
                timestamp=datetime.now().isoformat(),
                processing_time_ms=(time.time() - start_time) * 1000
            )
        
        # Step 1: Detect faces
        (h, w) = image.shape[:2]
        blob = cv2.dnn.blobFromImage(
            cv2.resize(image, (300, 300)),
            1.0,
            (300, 300),
            (104.0, 177.0, 123.0)
        )
        
        face_net.setInput(blob)
        detections = face_net.forward()
        
        # Step 2: Recognize each face
        recognized_persons = []
        
        for i in range(detections.shape[2]):
            confidence = float(detections[0, 0, i, 2])
            
            if confidence > CONFIDENCE_THRESHOLD:
                box = detections[0, 0, i, 3:7] * np.array([w, h, w, h])
                (x1, y1, x2, y2) = box.astype("int")
                
                # Ensure coordinates are valid
                x1 = max(0, x1)
                y1 = max(0, y1)
                x2 = min(w, x2)
                y2 = min(h, y2)
                
                # Extract and resize face
                face_roi = image[y1:y2, x1:x2]
                
                if face_roi.shape[0] > 0 and face_roi.shape[1] > 0:
                    face_resized = cv2.resize(face_roi, (160, 160))
                    
                    # Generate embedding
                    embedding = generate_embedding(face_resized)
                    
                    if embedding is not None:
                        # Match face
                        match_result = matcher.match_face(embedding)
                        
                        # Get person info
                        person_id = match_result['person_id']
                        similarity = match_result['similarity']
                        
                        if match_result['is_match']:
                            person_data = face_db.get_person(person_id)
                            person_name = person_data['name'] if person_data else "Unknown"
                        else:
                            person_name = "Unknown"
                            person_id = "unknown"
                        
                        recognized_persons.append(RecognizedPerson(
                            person_id=person_id,
                            person_name=person_name,
                            similarity=similarity,
                            bbox=FaceBox(
                                x1=x1,
                                y1=y1,
                                x2=x2,
                                y2=y2,
                                confidence=confidence
                            )
                        ))
        
        processing_time = time.time() - start_time
        
        return RecognizeResponse(
            status="success",
            persons=recognized_persons,
            timestamp=datetime.now().isoformat(),
            processing_time_ms=processing_time * 1000
        )
    
    except HTTPException:
        raise
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Recognition failed: {str(e)}")

print("✅ Recognition endpoint created!")
print("   POST /api/v1/recognize")

print("\n✅ Exercise 2.3 Complete!")
print("=" * 80)


EXERCISE 2.3: Recognition Endpoint

⏱️ Creating recognition endpoint...

✅ Recognition endpoint created!
   POST /api/v1/recognize

✅ Exercise 2.3 Complete!


In [13]:
print("\n" + "=" * 80)
print("👥 PART 3: ALERT & FACE MANAGEMENT ENDPOINTS")
print("=" * 80)


👥 PART 3: ALERT & FACE MANAGEMENT ENDPOINTS


In [14]:
# ==================================================
# EXERCISE 3.1: FACE MANAGEMENT ENDPOINTS
# ==================================================

print("\n" + "=" * 80)
print("EXERCISE 3.1: Face Management Endpoints")
print("=" * 80)

"""
📖 THEORY: Face Database API

Endpoints for managing face database:
- GET /api/v1/faces - List all persons
- POST /api/v1/faces - Add new person
- GET /api/v1/faces/{person_id} - Get person details
- DELETE /api/v1/faces/{person_id} - Remove person
"""

print("\n⏱️ Creating face management endpoints...\n")

# ==================================================
# 1. LIST ALL PERSONS
# ==================================================

@app.get("/api/v1/faces", response_model=ListPersonsResponse)
async def list_persons():
    """
    Get list of all known persons in database.
    
    Returns:
        ListPersonsResponse with all persons
    """
    try:
        persons_list = []
        
        for person_id, person_data in face_db.get_all_persons().items():
            persons_list.append(PersonInfo(
                person_id=person_id,
                name=person_data.get('name', 'Unknown'),
                face_count=len(person_data.get('faces', [])),
                added_date=person_data.get('added_date', ''),
                metadata=person_data.get('metadata')
            ))
        
        return ListPersonsResponse(
            status="success",
            total_persons=len(persons_list),
            persons=persons_list
        )
    
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Failed to list persons: {str(e)}")

print("✅ List persons endpoint created!")
print("   GET /api/v1/faces")

# ==================================================
# 2. ADD NEW PERSON
# ==================================================

@app.post("/api/v1/faces", response_model=AddPersonResponse)
async def add_person(request: AddPersonRequest):
    """
    Add new person to face database.
    
    Args:
        request: AddPersonRequest with name and face image
    
    Returns:
        AddPersonResponse with person_id
    """
    try:
        # Decode image
        image = base64_to_image(request.image)
        validate_image_size(image)
        
        # Generate person ID
        person_id = request.name.lower().replace(' ', '_')
        
        # Check if person already exists
        if face_db.get_person(person_id) is not None:
            raise HTTPException(
                status_code=400,
                detail=f"Person '{request.name}' already exists in database"
            )
        
        # Detect face in image
        if face_net is None:
            raise HTTPException(status_code=503, detail="Face detection model not loaded")
        
        (h, w) = image.shape[:2]
        blob = cv2.dnn.blobFromImage(
            cv2.resize(image, (300, 300)),
            1.0,
            (300, 300),
            (104.0, 177.0, 123.0)
        )
        
        face_net.setInput(blob)
        detections = face_net.forward()
        
        # Find best face
        best_confidence = 0
        best_face = None
        
        for i in range(detections.shape[2]):
            confidence = float(detections[0, 0, i, 2])
            
            if confidence > best_confidence:
                best_confidence = confidence
                box = detections[0, 0, i, 3:7] * np.array([w, h, w, h])
                (x1, y1, x2, y2) = box.astype("int")
                
                x1 = max(0, x1)
                y1 = max(0, y1)
                x2 = min(w, x2)
                y2 = min(h, y2)
                
                face_roi = image[y1:y2, x1:x2]
                if face_roi.shape[0] > 0 and face_roi.shape[1] > 0:
                    best_face = cv2.resize(face_roi, (160, 160))
        
        if best_face is None:
            raise HTTPException(status_code=400, detail="No face detected in image")
        
        # Generate embedding
        embedding = generate_embedding(best_face)
        
        if embedding is None:
            raise HTTPException(status_code=500, detail="Failed to generate face embedding")
        
        # Add to database
        face_db.add_person(person_id, request.name, embedding)
        
        # Add to matcher
        matcher.add_person(person_id, [embedding])
        
        return AddPersonResponse(
            status="success",
            person_id=person_id,
            message=f"Person '{request.name}' added successfully"
        )
    
    except HTTPException:
        raise
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Failed to add person: {str(e)}")

print("✅ Add person endpoint created!")
print("   POST /api/v1/faces")

# ==================================================
# 3. GET PERSON DETAILS
# ==================================================

@app.get("/api/v1/faces/{person_id}")
async def get_person(person_id: str):
    """
    Get details of specific person.
    
    Args:
        person_id: Person identifier
    
    Returns:
        Person information
    """
    try:
        person_data = face_db.get_person(person_id)
        
        if person_data is None:
            raise HTTPException(status_code=404, detail=f"Person '{person_id}' not found")
        
        return {
            "status": "success",
            "person": PersonInfo(
                person_id=person_id,
                name=person_data.get('name', 'Unknown'),
                face_count=len(person_data.get('faces', [])),
                added_date=person_data.get('added_date', ''),
                metadata=person_data.get('metadata')
            )
        }
    
    except HTTPException:
        raise
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Failed to get person: {str(e)}")

print("✅ Get person endpoint created!")
print("   GET /api/v1/faces/{person_id}")

# ==================================================
# 4. DELETE PERSON
# ==================================================

@app.delete("/api/v1/faces/{person_id}")
async def delete_person(person_id: str):
    """
    Remove person from database.
    
    Args:
        person_id: Person identifier
    
    Returns:
        Success message
    """
    try:
        person_data = face_db.get_person(person_id)
        
        if person_data is None:
            raise HTTPException(status_code=404, detail=f"Person '{person_id}' not found")
        
        # Remove from database
        del face_db.database["persons"][person_id]
        
        # Remove from matcher
        if person_id in matcher.database:
            del matcher.database[person_id]
        
        return {
            "status": "success",
            "message": f"Person '{person_id}' removed successfully"
        }
    
    except HTTPException:
        raise
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Failed to delete person: {str(e)}")

print("✅ Delete person endpoint created!")
print("   DELETE /api/v1/faces/{person_id}")

print("\n✅ All face management endpoints created!")
print("   GET    /api/v1/faces")
print("   POST   /api/v1/faces")
print("   GET    /api/v1/faces/{person_id}")
print("   DELETE /api/v1/faces/{person_id}")

print("\n✅ Exercise 3.1 Complete!")
print("=" * 80)


EXERCISE 3.1: Face Management Endpoints

⏱️ Creating face management endpoints...

✅ List persons endpoint created!
   GET /api/v1/faces
✅ Add person endpoint created!
   POST /api/v1/faces
✅ Get person endpoint created!
   GET /api/v1/faces/{person_id}
✅ Delete person endpoint created!
   DELETE /api/v1/faces/{person_id}

✅ All face management endpoints created!
   GET    /api/v1/faces
   POST   /api/v1/faces
   GET    /api/v1/faces/{person_id}
   DELETE /api/v1/faces/{person_id}

✅ Exercise 3.1 Complete!


In [15]:
# ==================================================
# EXERCISE 3.2: ALERT MANAGEMENT ENDPOINTS
# ==================================================

print("\n" + "=" * 80)
print("EXERCISE 3.2: Alert Management Endpoints")
print("=" * 80)

"""
📖 THEORY: Alert Management API

Endpoints for managing alerts:
- GET /api/v1/alerts - List all alerts
- POST /api/v1/alerts/acknowledge - Acknowledge alert
"""

print("\n⏱️ Creating alert management endpoints...\n")

# Simple in-memory alert storage (will be replaced with database in Day 37)
alerts_storage = []

# ==================================================
# 1. LIST ALERTS
# ==================================================

@app.get("/api/v1/alerts", response_model=ListAlertsResponse)
async def list_alerts(
    limit: Optional[int] = 50,
    priority: Optional[str] = None,
    acknowledged: Optional[bool] = None
):
    """
    Get list of alerts with optional filters.
    
    Args:
        limit: Maximum number of alerts to return
        priority: Filter by priority (critical, high, medium, low)
        acknowledged: Filter by acknowledgment status
    
    Returns:
        ListAlertsResponse with alerts
    """
    try:
        # Filter alerts
        filtered_alerts = alerts_storage.copy()
        
        if priority:
            filtered_alerts = [a for a in filtered_alerts if a.get('priority') == priority]
        
        if acknowledged is not None:
            filtered_alerts = [a for a in filtered_alerts if a.get('acknowledged') == acknowledged]
        
        # Limit results
        filtered_alerts = filtered_alerts[:limit]
        
        # Convert to AlertInfo objects
        alerts_list = []
        for alert in filtered_alerts:
            alerts_list.append(AlertInfo(
                alert_id=alert.get('alert_id', 0),
                timestamp=alert.get('timestamp', ''),
                alert_type=alert.get('alert_type', ''),
                priority=alert.get('priority', ''),
                person_id=alert.get('person_id', ''),
                person_name=alert.get('person_name', ''),
                description=alert.get('description', ''),
                acknowledged=alert.get('acknowledged', False)
            ))
        
        return ListAlertsResponse(
            status="success",
            total_alerts=len(alerts_list),
            alerts=alerts_list
        )
    
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Failed to list alerts: {str(e)}")

print("✅ List alerts endpoint created!")
print("   GET /api/v1/alerts")

# ==================================================
# 2. ACKNOWLEDGE ALERT
# ==================================================

@app.post("/api/v1/alerts/acknowledge", response_model=AcknowledgeAlertResponse)
async def acknowledge_alert(request: AcknowledgeAlertRequest):
    """
    Acknowledge an alert.
    
    Args:
        request: AcknowledgeAlertRequest with alert_id
    
    Returns:
        AcknowledgeAlertResponse with success message
    """
    try:
        # Find alert
        alert = None
        for a in alerts_storage:
            if a.get('alert_id') == request.alert_id:
                alert = a
                break
        
        if alert is None:
            raise HTTPException(status_code=404, detail=f"Alert {request.alert_id} not found")
        
        # Update alert
        alert['acknowledged'] = True
        alert['acknowledged_by'] = request.acknowledged_by
        alert['acknowledged_at'] = datetime.now().isoformat()
        if request.notes:
            alert['notes'] = request.notes
        
        return AcknowledgeAlertResponse(
            status="success",
            message=f"Alert {request.alert_id} acknowledged successfully"
        )
    
    except HTTPException:
        raise
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Failed to acknowledge alert: {str(e)}")

print("✅ Acknowledge alert endpoint created!")
print("   POST /api/v1/alerts/acknowledge")

# ==================================================
# 3. CREATE ALERT (HELPER FUNCTION)
# ==================================================

def create_alert(
    alert_type: str,
    priority: str,
    person_id: str,
    person_name: str,
    description: str
) -> int:
    """
    Create new alert in storage.
    
    Args:
        alert_type: Type of alert
        priority: Alert priority
        person_id: Person ID
        person_name: Person name
        description: Alert description
    
    Returns:
        Alert ID
    """
    alert_id = len(alerts_storage) + 1
    
    alert = {
        'alert_id': alert_id,
        'timestamp': datetime.now().isoformat(),
        'alert_type': alert_type,
        'priority': priority,
        'person_id': person_id,
        'person_name': person_name,
        'description': description,
        'acknowledged': False
    }
    
    alerts_storage.append(alert)
    
    return alert_id

print("✅ Create alert helper function added!")

# Add some dummy alerts for testing
if len(alerts_storage) == 0:
    create_alert(
        alert_type="unknown_person",
        priority="critical",
        person_id="unknown_1",
        person_name="Unknown Person",
        description="Unknown person detected at main entrance"
    )
    create_alert(
        alert_type="after_hours",
        priority="high",
        person_id="audrey",
        person_name="Audrey",
        description="Access after hours detected"
    )
    print(f"\n💡 Added {len(alerts_storage)} demo alerts for testing")

print("\n✅ All alert management endpoints created!")
print("   GET  /api/v1/alerts")
print("   POST /api/v1/alerts/acknowledge")

print("\n✅ Exercise 3.2 Complete!")
print("=" * 80)


EXERCISE 3.2: Alert Management Endpoints

⏱️ Creating alert management endpoints...

✅ List alerts endpoint created!
   GET /api/v1/alerts
✅ Acknowledge alert endpoint created!
   POST /api/v1/alerts/acknowledge
✅ Create alert helper function added!

💡 Added 2 demo alerts for testing

✅ All alert management endpoints created!
   GET  /api/v1/alerts
   POST /api/v1/alerts/acknowledge

✅ Exercise 3.2 Complete!


In [16]:
# ==================================================
# EXERCISE 3.3: STATISTICS ENDPOINT
# ==================================================

print("\n" + "=" * 80)
print("EXERCISE 3.3: Statistics Endpoint")
print("=" * 80)

"""
📖 THEORY: System Statistics API

GET /api/v1/stats - Get system statistics
"""

print("\n⏱️ Creating statistics endpoint...\n")

@app.get("/api/v1/stats")
async def get_statistics():
    """
    Get system statistics.
    
    Returns:
        System statistics
    """
    try:
        # Count alerts by priority
        priority_counts = {
            'critical': 0,
            'high': 0,
            'medium': 0,
            'low': 0
        }
        
        for alert in alerts_storage:
            priority = alert.get('priority', 'low')
            if priority in priority_counts:
                priority_counts[priority] += 1
        
        # Count acknowledged vs unacknowledged
        acknowledged_count = sum(1 for a in alerts_storage if a.get('acknowledged', False))
        unacknowledged_count = len(alerts_storage) - acknowledged_count
        
        return {
            "status": "success",
            "timestamp": datetime.now().isoformat(),
            "statistics": {
                "database": {
                    "total_persons": len(face_db.get_all_persons()),
                    "total_faces": sum(len(p.get('faces', [])) for p in face_db.get_all_persons().values())
                },
                "alerts": {
                    "total_alerts": len(alerts_storage),
                    "acknowledged": acknowledged_count,
                    "unacknowledged": unacknowledged_count,
                    "by_priority": priority_counts
                },
                "system": {
                    "models_loaded": face_net is not None,
                    "embedding_model_loaded": True,
                    "matcher_loaded": True,
                    "database_loaded": True
                }
            }
        }
    
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Failed to get statistics: {str(e)}")

print("✅ Statistics endpoint created!")
print("   GET /api/v1/stats")

print("\n✅ Exercise 3.3 Complete!")
print("=" * 80)


EXERCISE 3.3: Statistics Endpoint

⏱️ Creating statistics endpoint...

✅ Statistics endpoint created!
   GET /api/v1/stats

✅ Exercise 3.3 Complete!


In [17]:
print("\n" + "=" * 80)
print("🧪 PART 4: TESTING, DOCUMENTATION & SUMMARY")
print("=" * 80)


🧪 PART 4: TESTING, DOCUMENTATION & SUMMARY


In [18]:
# ==================================================
# EXERCISE 4.1: API SUMMARY & ENDPOINTS LIST
# ==================================================

print("\n" + "=" * 80)
print("EXERCISE 4.1: API Summary & Endpoints List")
print("=" * 80)

"""
📖 COMPLETE API DOCUMENTATION

Summary of all endpoints created.
"""

print("\n📊 API ENDPOINTS SUMMARY")
print("=" * 80)

endpoints = {
    "Health & Status": [
        ("GET", "/", "Root endpoint - API status"),
        ("GET", "/api/v1/health", "Health check endpoint"),
        ("GET", "/api/v1/stats", "System statistics")
    ],
    "Face Detection & Recognition": [
        ("POST", "/api/v1/detect", "Detect faces in image"),
        ("POST", "/api/v1/recognize", "Recognize persons in image")
    ],
    "Face Database Management": [
        ("GET", "/api/v1/faces", "List all persons"),
        ("POST", "/api/v1/faces", "Add new person"),
        ("GET", "/api/v1/faces/{person_id}", "Get person details"),
        ("DELETE", "/api/v1/faces/{person_id}", "Remove person")
    ],
    "Alert Management": [
        ("GET", "/api/v1/alerts", "List alerts with filters"),
        ("POST", "/api/v1/alerts/acknowledge", "Acknowledge alert")
    ]
}

for category, endpoint_list in endpoints.items():
    print(f"\n📌 {category}:")
    for method, path, description in endpoint_list:
        print(f"   {method:6s} {path:40s} - {description}")

print("\n" + "=" * 80)
print("📊 TOTAL ENDPOINTS: 12")
print("=" * 80)

print("\n💡 API Documentation:")
print("   Swagger UI: http://localhost:8000/docs")
print("   ReDoc: http://localhost:8000/redoc")
print("   OpenAPI Schema: http://localhost:8000/openapi.json")

print("\n✅ Exercise 4.1 Complete!")
print("=" * 80)


EXERCISE 4.1: API Summary & Endpoints List

📊 API ENDPOINTS SUMMARY

📌 Health & Status:
   GET    /                                        - Root endpoint - API status
   GET    /api/v1/health                           - Health check endpoint
   GET    /api/v1/stats                            - System statistics

📌 Face Detection & Recognition:
   POST   /api/v1/detect                           - Detect faces in image
   POST   /api/v1/recognize                        - Recognize persons in image

📌 Face Database Management:
   GET    /api/v1/faces                            - List all persons
   POST   /api/v1/faces                            - Add new person
   GET    /api/v1/faces/{person_id}                - Get person details
   DELETE /api/v1/faces/{person_id}                - Remove person

📌 Alert Management:
   GET    /api/v1/alerts                           - List alerts with filters
   POST   /api/v1/alerts/acknowledge               - Acknowledge alert

📊 TOTAL ENDPOINTS: 1

In [19]:
# ==================================================
# EXERCISE 4.2: START API SERVER
# ==================================================

print("\n" + "=" * 80)
print("EXERCISE 4.2: Start API Server")
print("=" * 80)

"""
📖 THEORY: Running FastAPI Server

Use uvicorn to run the API server.
"""

print("\n⏱️ API Server Instructions\n")

print("🚀 TO START THE API SERVER:")
print("=" * 80)

print("""
Option 1: Run from Jupyter (Background Process)
-----------------------------------------------
This will run the API server in the background while you continue in Jupyter:

import threading

def run_api():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")

# Start in background thread
api_thread = threading.Thread(target=run_api, daemon=True)
api_thread.start()

print("✅ API Server started in background!")
print("📊 Access Swagger UI: http://localhost:8000/docs")


Option 2: Run from Command Line (Recommended)
---------------------------------------------
Open a new terminal and run:

cd week6_api_dashboard_deployment
uvicorn day36_backend_api:app --reload --host 0.0.0.0 --port 8000

The --reload flag auto-restarts when you change code.


Option 3: Run Directly (Blocking)
---------------------------------
This will block the notebook until you stop it (Ctrl+C):

uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")
""")

print("\n🔗 AFTER STARTING:")
print("=" * 80)
print("   API Root: http://localhost:8000")
print("   Swagger UI: http://localhost:8000/docs")
print("   ReDoc: http://localhost:8000/redoc")
print("   OpenAPI Schema: http://localhost:8000/openapi.json")

print("\n💡 TESTING THE API:")
print("=" * 80)
print("""
1. Open browser → http://localhost:8000/docs
2. Try the interactive API documentation
3. Test endpoints directly from Swagger UI
4. See request/response examples
""")

print("\n✅ Exercise 4.2 Complete!")
print("=" * 80)


EXERCISE 4.2: Start API Server

⏱️ API Server Instructions

🚀 TO START THE API SERVER:

Option 1: Run from Jupyter (Background Process)
-----------------------------------------------
This will run the API server in the background while you continue in Jupyter:

import threading

def run_api():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")

# Start in background thread
api_thread = threading.Thread(target=run_api, daemon=True)
api_thread.start()

print("✅ API Server started in background!")
print("📊 Access Swagger UI: http://localhost:8000/docs")


Option 2: Run from Command Line (Recommended)
---------------------------------------------
Open a new terminal and run:

cd week6_api_dashboard_deployment
uvicorn day36_backend_api:app --reload --host 0.0.0.0 --port 8000

The --reload flag auto-restarts when you change code.


Option 3: Run Directly (Blocking)
---------------------------------
This will block the notebook until you stop it (Ctrl+C):

uvicorn.run(app, ho

In [20]:
# ==================================================
# EXERCISE 4.3: TEST API WITH PYTHON REQUESTS
# ==================================================

print("\n" + "=" * 80)
print("EXERCISE 4.3: Test API with Python Requests")
print("=" * 80)

"""
📖 THEORY: Testing API Programmatically

Use requests library to test API endpoints.
"""

print("\n⏱️ Creating API test functions...\n")

import requests

API_BASE_URL = "http://localhost:8000"

def test_health_check():
    """Test health check endpoint."""
    print("🧪 Testing: GET /api/v1/health")
    
    try:
        response = requests.get(f"{API_BASE_URL}/api/v1/health", timeout=5)
        
        if response.status_code == 200:
            data = response.json()
            print(f"   ✅ Status: {response.status_code}")
            print(f"   ✅ Response: {data}")
            return True
        else:
            print(f"   ❌ Status: {response.status_code}")
            return False
    
    except requests.exceptions.ConnectionError:
        print("   ⚠️  Connection failed - Is the API server running?")
        print("   💡 Start the server first (see Exercise 4.2)")
        return False
    except Exception as e:
        print(f"   ❌ Error: {e}")
        return False

def test_list_faces():
    """Test list faces endpoint."""
    print("\n🧪 Testing: GET /api/v1/faces")
    
    try:
        response = requests.get(f"{API_BASE_URL}/api/v1/faces", timeout=5)
        
        if response.status_code == 200:
            data = response.json()
            print(f"   ✅ Status: {response.status_code}")
            print(f"   ✅ Total persons: {data.get('total_persons', 0)}")
            return True
        else:
            print(f"   ❌ Status: {response.status_code}")
            return False
    
    except requests.exceptions.ConnectionError:
        print("   ⚠️  Connection failed - Is the API server running?")
        return False
    except Exception as e:
        print(f"   ❌ Error: {e}")
        return False

def test_list_alerts():
    """Test list alerts endpoint."""
    print("\n🧪 Testing: GET /api/v1/alerts")
    
    try:
        response = requests.get(f"{API_BASE_URL}/api/v1/alerts", timeout=5)
        
        if response.status_code == 200:
            data = response.json()
            print(f"   ✅ Status: {response.status_code}")
            print(f"   ✅ Total alerts: {data.get('total_alerts', 0)}")
            return True
        else:
            print(f"   ❌ Status: {response.status_code}")
            return False
    
    except requests.exceptions.ConnectionError:
        print("   ⚠️  Connection failed - Is the API server running?")
        return False
    except Exception as e:
        print(f"   ❌ Error: {e}")
        return False

def test_statistics():
    """Test statistics endpoint."""
    print("\n🧪 Testing: GET /api/v1/stats")
    
    try:
        response = requests.get(f"{API_BASE_URL}/api/v1/stats", timeout=5)
        
        if response.status_code == 200:
            data = response.json()
            print(f"   ✅ Status: {response.status_code}")
            print(f"   ✅ Statistics retrieved")
            if 'statistics' in data:
                stats = data['statistics']
                print(f"      - Persons: {stats['database']['total_persons']}")
                print(f"      - Alerts: {stats['alerts']['total_alerts']}")
            return True
        else:
            print(f"   ❌ Status: {response.status_code}")
            return False
    
    except requests.exceptions.ConnectionError:
        print("   ⚠️  Connection failed - Is the API server running?")
        return False
    except Exception as e:
        print(f"   ❌ Error: {e}")
        return False

def run_all_tests():
    """Run all API tests."""
    print("\n" + "=" * 80)
    print("🧪 RUNNING ALL API TESTS")
    print("=" * 80)
    
    results = []
    
    results.append(("Health Check", test_health_check()))
    results.append(("List Faces", test_list_faces()))
    results.append(("List Alerts", test_list_alerts()))
    results.append(("Statistics", test_statistics()))
    
    print("\n" + "=" * 80)
    print("📊 TEST RESULTS SUMMARY")
    print("=" * 80)
    
    for test_name, passed in results:
        status = "✅ PASSED" if passed else "❌ FAILED"
        print(f"   {test_name:20s} {status}")
    
    total = len(results)
    passed = sum(1 for _, p in results if p)
    
    print(f"\n   Total: {passed}/{total} tests passed")
    
    if passed == total:
        print("\n   🎉 ALL TESTS PASSED!")
    else:
        print("\n   ⚠️  Some tests failed - check API server")

print("✅ Test functions created!")
print("\n💡 To run tests:")
print("   1. Start the API server (see Exercise 4.2)")
print("   2. Run: run_all_tests()")

print("\n✅ Exercise 4.3 Complete!")
print("=" * 80)


EXERCISE 4.3: Test API with Python Requests

⏱️ Creating API test functions...

✅ Test functions created!

💡 To run tests:
   1. Start the API server (see Exercise 4.2)
   2. Run: run_all_tests()

✅ Exercise 4.3 Complete!
